# 🤖 Ollama dans le cloud — Google Colab (gratuit, GPU T4)

Ce notebook transforme une session Colab **gratuite** en vrai serveur **Ollama**, accessible
depuis Internet. Vous obtenez une URL `https://…trycloudflare.com` à coller dans votre site
**Ollama Web** (variable `OLLAMA_REMOTE_URL`) ou dans n'importe quel client compatible Ollama/OpenAI.

**Avant de commencer** : menu `Exécution` → `Modifier le type d'exécution` → **T4 GPU**.

> ⏱️ Comptez 5 à 10 minutes la première fois (téléchargement du modèle).
> Colab gratuit coupe la session après ~90 min d'inactivité, et l'URL du tunnel change à chaque
> redémarrage : pour un service permanent, préférez Ollama Cloud ou un petit VPS.

In [ ]:
# 1️⃣ Vérifier le GPU (si erreur : activez le T4 dans « Modifier le type d'exécution »)
!nvidia-smi

In [ ]:
# 2️⃣ Installer Ollama et démarrer le serveur (en arrière-plan)
!curl -fsSL https://ollama.com/install.sh | sh

import os, subprocess, time
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'   # écoute sur toutes les interfaces
subprocess.Popen(['ollama', 'serve'], stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)

for i in range(60):
    time.sleep(1)
    try:
        import urllib.request
        print('✔ Ollama prêt :', urllib.request.urlopen('http://127.0.0.1:11434/api/version').read().decode())
        break
    except Exception:
        pass
else:
    print('⚠️ Ollama ne répond pas — regardez /tmp/ollama.log')

## 📦 Choisir un modèle qui tient dans le T4 (16 Go de VRAM)

| Modèle | Taille | Remarque |
|---|---|---|
| `qwen2.5:3b` | ≈2 Go | très rapide, idéal pour tester |
| `gemma3:4b` | ≈3,3 Go | multimodal (images) |
| `llama3.2:3b` | ≈2 Go | polyvalent |
| `qwen3:8b` | ≈5 Go | bon compromis qualité/vitesse |
| `llama3.1:8b` | ≈4,9 Go | classique |
| `gemma3:12b` | ≈8 Go | plus lent mais plus fin |
| `qwen3:14b` | ≈9 Go | la limite raisonnable du T4 |

Modifiez la variable `MODELE` ci-dessous puis exécutez la cellule.

In [ ]:
# 3️⃣ Télécharger le modèle (5–15 min la première fois)
MODELE = 'qwen2.5:3b'   # ← changez ici
!ollama pull {MODELE}
!ollama list

In [ ]:
# 4️⃣ Ouvrir un tunnel public sécurisé (Cloudflare, aucun compte requis)
!wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb
!dpkg -i /tmp/cloudflared.deb >/dev/null 2>&1 || true

import re, subprocess, time
log = open('/tmp/cloudflared.log', 'w')
subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:11434',
     '--http-host-header', 'localhost:11434', '--no-autoupdate'],
    stdout=log, stderr=subprocess.STDOUT,
)

url = None
for _ in range(45):
    time.sleep(2)
    try:
        txt = open('/tmp/cloudflared.log').read()
    except FileNotFoundError:
        continue
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', txt)
    if m:
        url = m.group(0)
        break

if url:
    print('\n' + '=' * 68)
    print('  ✅ VOTRE SERVEUR OLLAMA EST EN LIGNE')
    print('=' * 68)
    print(f'  URL de base (à coller dans Ollama Web) : {url}')
    print(f'  Test rapide : curl {url}/api/tags')
    print('=' * 68)
else:
    print('⚠️ Tunnel non détecté — relancez cette cellule ou consultez /tmp/cloudflared.log')

## 🔌 Brancher le site web sur cette session Colab

**Option A — Render + Colab** : dans les variables d'environnement de votre service Render
(`ollama-web`), mettez `OLLAMA_REMOTE_URL` = l'URL affichée ci-dessus, puis redéploiement / redémarrage.

**Option B — en local** : dans `ollama-web/.env` :

```bash
OLLAMA_REMOTE_URL=https://votre-url.trycloudflare.com
OLLAMA_REMOTE_LABEL=Colab T4
DEFAULT_PROVIDER=remote
```
puis `npm start` et ouvrez http://localhost:3000 — le menu « Modèle » liste les modèles du T4.

**Client OpenAI compatible** (LangChain, Open WebUI…) : base = `<URL>/v1`, clé au choix.

**Astuce** : pour rafraîchir l'URL à la volée depuis le site, la variable peut être changée sans
redémarrer Render si vous passez par le tableau de bord (Environment → Save → redéploiement auto).

### 🧯 En cas de problème

| Symptôme | Solution |
|---|---|
| `nvidia-smi` introuvable | Réactivez le GPU : `Modifier le type d'exécution` → T4 |
| Le site affiche « injoignable » | Vérifiez que la cellule 4 tourne toujours ; l'URL change à chaque session |
| Réponses très lentes | Le modèle est trop gros pour un T4 : passez à `qwen2.5:3b` |
| Erreur 403 renvoyée par Ollama | Le tunnel doit être lancé avec `--http-host-header localhost:11434` |
| Session coupée | Colab gratuit limite les sessions ; relancez les cellules et mettez l'URL à jour |